In [0]:
from pyspark.sql.functions import *

hub_customer = spark.table("default.hub_customer")
hub_order = spark.table("default.hub_order")
hub_product = spark.table("default.hub_product")

raw_orders = spark.table("default.raw_orders")
raw_order_details = spark.table("default.raw_order_details")

In [0]:
hub_customer.printSchema()

hub_order.printSchema()

hub_product.printSchema()

raw_orders.printSchema()

raw_order_details.printSchema()

root
 |-- Customer_HK: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)

root
 |-- Order_HK: string (nullable = true)
 |-- OrderID: string (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)

root
 |-- Product_HK: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)

root
 |-- OrderID: string (nullable = true)
 |-- CustID: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Batch_ID: integer (nullable = true)

root
 |-- OrderID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Recor

In [0]:
customer_order = raw_orders.alias("o") \
.join(
    hub_customer.alias("c"),
    col("o.CustID") == col("c.Customer_ID"),
    "inner"
) \
.join(
    hub_order.alias("h"),
    col("o.OrderID") == col("h.OrderID"),
    "inner"
)

In [0]:
customer_order.show(5, False)

+-------+------+------+----------+-------------------------+-------------+--------+----------------------------------------------------------------+-----------+--------------------------+-------------+----------------------------------------------------------------+-------+-------------------------+-------------+
|OrderID|CustID|Amount|OrderDate |Load_Date                |Record_Source|Batch_ID|Customer_HK                                                     |Customer_ID|Load_Date                 |Record_Source|Order_HK                                                        |OrderID|Load_Date                |Record_Source|
+-------+------+------+----------+-------------------------+-------------+--------+----------------------------------------------------------------+-----------+--------------------------+-------------+----------------------------------------------------------------+-------+-------------------------+-------------+
|O122   |C023  |617.0 |2024-01-23|2026-07-13 10:07:07.1

In [0]:
from pyspark.sql.functions import *

link_customer_order = customer_order.select(

    sha2(
        concat_ws(
            "|",
            col("c.Customer_HK"),
            col("h.Order_HK")
        ),
        256
    ).alias("Link_Customer_Order_HK"),

    col("c.Customer_HK").alias("Customer_HK"),

    col("h.Order_HK").alias("Order_HK"),

    col("o.Load_Date").alias("Load_Date"),

    col("o.Record_Source").alias("Record_Source")

).dropDuplicates()

In [0]:
link_customer_order.show(5, False)

+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+-------------------------+-------------+
|Link_Customer_Order_HK                                          |Customer_HK                                                     |Order_HK                                                        |Load_Date                |Record_Source|
+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+-------------------------+-------------+
|6af8d874ed61be4069c7a6ae7c7ec827cc04c6ebd7ecfe97811b4e7d810730e0|b33af7c2172dc903ee024904a1fa429bdd860f3820d481e67226d44154b4dc72|7afa854782e265f397edc5636853ddda8e9698472d89d1021f77e6943a65ef9e|2026-07-13 10:07:07.11227|Orders_CSV   |
|6d9447e7054b2d752dac0a3f401c0aadadea188d8f6dffe1c37

In [0]:
%sql
DROP TABLE IF EXISTS default.link_customer_order;

In [0]:
link_customer_order.write.mode("overwrite").saveAsTable("default.link_customer_order")

In [0]:
%sql
SELECT COUNT(*) FROM default.link_customer_order;

COUNT(*)
500


In [0]:
order_product = raw_order_details.alias("d") \
.join(
    hub_order.alias("o"),
    col("d.OrderID") == col("o.OrderID"),
    "inner"
) \
.join(
    hub_product.alias("p"),
    col("d.ProductID") == col("p.ProductID"),
    "inner"
)

In [0]:
order_product.show(5, False)

+-------+---------+--------+--------------------------+----------------+--------+----------------------------------------------------------------+-------+-------------------------+-------------+----------------------------------------------------------------+---------+-------------------------+-------------+
|OrderID|ProductID|Quantity|Load_Date                 |Record_Source   |Batch_ID|Order_HK                                                        |OrderID|Load_Date                |Record_Source|Product_HK                                                      |ProductID|Load_Date                |Record_Source|
+-------+---------+--------+--------------------------+----------------+--------+----------------------------------------------------------------+-------+-------------------------+-------------+----------------------------------------------------------------+---------+-------------------------+-------------+
|O122   |P003     |3       |2026-07-13 10:08:58.928252|OrderDetails_CS

In [0]:
link_order_product = order_product.select(

    sha2(
        concat_ws(
            "|",
            col("o.Order_HK"),
            col("p.Product_HK")
        ),
        256
    ).alias("Link_Order_Product_HK"),

    col("o.Order_HK").alias("Order_HK"),

    col("p.Product_HK").alias("Product_HK"),

    col("d.Load_Date").alias("Load_Date"),

    col("d.Record_Source").alias("Record_Source")

).dropDuplicates()

In [0]:
%sql
DROP TABLE IF EXISTS default.link_order_product;

In [0]:
link_order_product.write.mode("overwrite").saveAsTable("default.link_order_product")

In [0]:
%sql
SELECT COUNT(*) FROM default.link_customer_order;

SELECT COUNT(*) FROM default.link_order_product;

COUNT(*)
500
